In [ ]:
from preprocess_data import clean_connect_four, build_dataset
from cnn_model import train_value_net
import pandas as pd
import numpy as np
import time

In [ ]:
# Clean data
raw = pd.read_csv("data/connect_four.csv").to_numpy(dtype=np.int8)
result = clean_connect_four(raw)

# Save
np.savetxt("data/connect_four_clean.csv",
        np.column_stack([result.boards.reshape(-1, 42), result.winners]),
        fmt="%d", delimiter=",")
np.save("data/boards_clean.npy", result.boards)
np.save("data/winners_clean.npy", result.winners)
result.audit.to_parquet("data/preprocess_audit.parquet", index=False)

In [ ]:
# Load cleaned data
boards = np.load("data/boards_clean.npy")
winners = np.load("data/winners_clean.npy")
print(f"{len(boards):,} cleaned games")

# Track time and build the data to train the CNNe
t0 = time.perf_counter()
keys, values, counts, failed = build_dataset(boards, winners)
print(f"\nbuilt in {time.perf_counter() - t0:.1f}s")

# Cache positions
np.savez_compressed(
    "data/positions.npz", keys=keys, values=values, counts=counts
)
print(f"wrote positions.npz  ({len(keys):,} positions)")

# Cache failed to reconstruct boards
rows = np.column_stack([
    np.asarray(boards)[failed].reshape(len(failed), 42),
    np.asarray(winners)[failed],
])

np.savetxt("data/failed.csv", rows, fmt="%d", delimiter=",")
print(f"wrote failed.csv ({len(rows)} boards)")

In [ ]:
# Train a model
model, history = train_value_net(
    data="data/positions.npz", 
    out="models/value_net.pt",
    epochs=20, 
    filters=64, 
    blocks=3, 
    lr=1e-3
)

# Plot performance
history.plot(x="epoch", y=["train", "val"], figsize=(7, 4), grid=True)